In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/apache1_dataset.csv")

print(df.shape)

(104557, 67)


In [11]:
drop_cols = [
    "ID",
    "UHID",
    "IPNumber",
    "ICUChartDate",

    "CreatedBy",
    "CreatedDate",
    "UpdatedBy",
    "UpdatedDate",

    "DOB",
    "AdmittingDoctor",
    "Ward",

    "RNK",
    "CREATEDON",
    "LOCATIONID",
    "DISCHARGEDATE",
    "PERIOD_WID",

    # APACHE-IV outputs (leakage)
    "ApacheivScore",
    "ApsScore",
    "EstimatedMortalityRate",

    # Hospital identifiers
    "UNIT_ID",
    "LOCATION",

    # Extra leakage / noisy columns
    "AdmissionDate",
    "APACHE_WARD"
]

los_df = df.drop(columns=drop_cols)

print(los_df.shape)

(104557, 44)


In [12]:
X = los_df.drop(
    columns=[
        "EstimatedLengthOfStay",
        "CUSTOMERSTATUS"
    ]
)

y = los_df["EstimatedLengthOfStay"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (104557, 42)
y Shape: (104557,)


In [13]:
print(X.columns.tolist())

['Age', 'Temperature', 'MeanArterialPressure', 'HeartRate', 'RespiratoryRate', 'FiO2', 'pO2', 'pCO2', 'ArterialpH', 'Sodium', 'UrineOutput', 'Creatinine', 'Urea', 'BSL', 'Albumin', 'Bilirubin', 'Hematocrit', 'WBC', 'IsGCSNotAvailable', 'GCSEyes', 'GCSVerbal', 'GCSMotor', 'MecanicalVentilation', 'CRF', 'Lymphoma', 'Cirrhosis', 'Leukemia', 'HepaticFailure', 'Immunosuppression', 'MetastaticCarcinoma', 'AIDS', 'PreICULengthOfStay', 'DiagnosisType', 'Origin', 'EmergencySurgery', 'Readmission', 'Thrombolysis', 'RespiratoryQuotient', 'AtmosphericPressure', 'SystemValue', 'DiagnosisValue', 'Gender']


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (83645, 42)
Test : (20912, 42)


In [15]:
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print(cat_cols)

['MecanicalVentilation', 'SystemValue', 'DiagnosisValue', 'Gender']


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols
        )
    ],
    remainder="passthrough"
)

In [17]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print("Model Ready")

Model Ready


In [18]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Pipeline Ready")

Pipeline Ready


In [19]:
pipeline.fit(X_train, y_train)

print("LOS Training Complete")

LOS Training Complete


In [20]:
preds = pipeline.predict(X_test)

print(preds[:10])

[2.6926842  2.9482694  2.6180036  1.3509129  5.7266874  4.016373
 2.7608438  5.3203406  3.7424464  0.81560117]


In [21]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds) ** 0.5
r2 = r2_score(y_test, preds)

print("=" * 60)
print("LOS MODEL RESULTS")
print("=" * 60)

print("MAE :", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R²  :", round(r2, 4))

LOS MODEL RESULTS
MAE : 0.328
RMSE: 0.459
R²  : 0.9466


In [22]:
import joblib

joblib.dump(
    pipeline,
    "../models/apache1_los_model.pkl"
)

print("LOS Model Saved Successfully")

LOS Model Saved Successfully


In [23]:
def los_to_days_hours(predicted_days):
    
    days = int(predicted_days)

    hours = round(
        (predicted_days - days) * 24
    )

    return f"{days} Days {hours} Hours"
